In [ ]:
# Full ML Cycle Habit Tracker (run this at the start of each project)

project_name = "SET_THIS_PROJECT_NAME"

cycle_steps = [
    "1_problem_definition",
    "2_data_collection_and_scope",
    "3_data_transformation_and_processing",
    "4_baseline_model_or_system",
    "5_evaluation_metrics_and_results",
    "6_error_analysis",
    "7_improvement_iteration",
    "8_deployment_or_handoff_notes",
]

status = {step: "TODO" for step in cycle_steps}
evidence = {step: "" for step in cycle_steps}

def mark_done(step, note=""):
    if step not in status:
        raise ValueError(f"Unknown step: {step}")
    status[step] = "DONE"
    evidence[step] = note

def show_progress():
    done_count = sum(1 for v in status.values() if v == "DONE")
    total = len(status)
    print(f"\nProject: {project_name}")
    print(f"Progress: {done_count}/{total} steps complete")
    for step in cycle_steps:
        note = f" | note: {evidence[step]}" if evidence[step] else ""
        print(f"- {step}: {status[step]}{note}")

# Example usage:
# mark_done("3_data_transformation_and_processing", "Handled missing values and encoded categories")
# mark_done("5_evaluation_metrics_and_results", "F1=0.81, recall=0.86")
show_progress()

# Full Process Lens For This Project

Use this section every time you start a project so you practice the same end-to-end workflow, not just isolated coding tasks.

## 1) Data Transformation and Processing (What and Why)

Raw data is rarely model-ready. Your first job is to transform data into a reliable, learnable format.

What to do in every project:
- Identify data types and expected schema.
- Handle missing values, duplicates, and inconsistent formats.
- Convert features into model-usable representations (encoding, scaling, tokenization, chunking, etc.).
- Keep transformations reproducible so train and inference use the same logic.
- Document assumptions and risks introduced by preprocessing choices.

Why this matters:
- Better preprocessing usually improves results more than switching algorithms.
- Poor preprocessing creates hidden errors that look like model failure.

## 2) Evaluating and Improving Models (What and Why)

Evaluation is not the last step. It is the loop that drives improvement.

What to do in every project:
- Start with a baseline and compare against it.
- Choose metrics tied to the real goal (not just convenience metrics).
- Inspect errors by slice (segments, classes, edge cases).
- Tune thresholds, features, prompts, retrieval settings, or hyperparameters based on evidence.
- Re-evaluate after each change and keep track of what improved and what regressed.

Why this matters:
- A model can appear good overall but fail on important cases.
- Iterative evaluation is how projects become production-ready, not just demo-ready.

## 3) Project Reflection Checklist

Before marking this project complete, confirm:
- I can explain how data was transformed and why.
- I can explain which metrics I chose and why.
- I can show at least one improvement cycle from evaluation findings.
- I can describe current limitations and next improvements.

# Week 4 — Data Visualization Deep Dive
> **Phase 0 | Foundation** — See your data before you model it.

---

## Beginner Start Here

**Visualization is not optional.** Every ML project starts with Exploratory Data Analysis (EDA), and EDA requires charts. You'll use charts to:
- Understand distributions
- Spot missing data and outliers
- Find relationships between features
- Communicate findings to stakeholders
- Debug model predictions

This week covers **matplotlib** (low-level, full control) and **seaborn** (high-level, beautiful statistical charts).

### What You Will Learn
- Matplotlib: line plots, scatter plots, bar charts, histograms, subplots
- Seaborn: distribution plots, correlation heatmap, categorical plots, pairplot
- How to label and style charts professionally
- The EDA workflow for any new dataset
- How to save figures for reports and portfolios

### Key Terms
| Term | Plain English |
|------|---------------|
| **Figure** | The entire canvas — the outermost container of your chart |
| **Axes** | A single chart (confusingly named — not the x/y axis themselves) |
| **Subplot** | Multiple Axes arranged in a grid within one Figure |
| **Distribution** | How frequently different values appear in a dataset |
| **Histogram** | A bar chart showing the distribution of a numeric column |
| **Scatter plot** | Dots showing the relationship between two numeric columns |
| **Correlation** | How much two variables move together (range: -1 to +1) |
| **Heatmap** | A grid of colored cells — each cell's color shows a value |
| **Box plot** | Shows median, quartiles, and outliers of a distribution |
| **KDE** | Kernel Density Estimate — a smooth version of a histogram |

In [ ]:
# ── First run check + imports ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import os

# Set a clean style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"✅ matplotlib {mpl.__version__}, seaborn {sns.__version__} loaded.")
print("Week 4 visualization notebook ready.")

In [ ]:
# ── Generate our practice dataset ────────────────────────────────────────────
# We'll use this for all visualizations in this notebook.

np.random.seed(42)
n = 300

tenure = np.random.randint(1, 72, n)
charges = np.random.uniform(20, 120, n).round(2)
agents = np.random.choice(["Contract", "Month-to-month", "One year"], n, p=[0.3, 0.5, 0.2])

churn_prob = 0.15 + 0.4*(charges > 90) + 0.3*(tenure < 10) - 0.2*(agents == "Contract")
churn_prob = np.clip(churn_prob, 0, 1)
churned = (np.random.rand(n) < churn_prob).astype(int)

df = pd.DataFrame({
    'tenure_months': tenure,
    'monthly_charges': charges,
    'contract': agents,
    'churned': churned,
    'support_tickets': np.random.poisson(2, n),
})
df['churn_label'] = df['churned'].map({0: 'Active', 1: 'Churned'})

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.1%}")
print(df.head())

---

## Section 1: matplotlib Fundamentals

**Two ways to use matplotlib:**
1. `plt.plot(...)` — quick and simple (pyplot API)
2. `fig, ax = plt.subplots()` then `ax.plot(...)` — full control (object-oriented API)

For anything beyond a single plot, always use the object-oriented API. It's the professional way.

In [ ]:
# ── matplotlib Basics: Line Plot ──────────────────────────────────────────────
# Use case: training loss curve over epochs

epochs = list(range(1, 21))
train_loss = [1.0 / (1 + 0.3*e) + np.random.rand() * 0.05 for e in epochs]
val_loss   = [1.2 / (1 + 0.25*e) + np.random.rand() * 0.08 for e in epochs]

# Object-oriented API (preferred)
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(epochs, train_loss, 'b-o', markersize=5, linewidth=2, label='Training Loss')
ax.plot(epochs, val_loss, 'r--s', markersize=5, linewidth=2, label='Validation Loss')

# Labels and formatting (always add these!)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training & Validation Loss Over Epochs', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(1, 20)

plt.tight_layout()
plt.show()

print("Key matplotlib elements:")
print("  fig  = the entire canvas")
print("  ax   = the single chart within the canvas")
print("  'b-o' = blue, solid line, circle markers")
print("  'r--s' = red, dashed line, square markers")

In [ ]:
# ── Histogram ────────────────────────────────────────────────────────────────
# Use case: understand the distribution of a numeric feature

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: distribution of monthly charges
axes[0].hist(df['monthly_charges'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['monthly_charges'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: ${df['monthly_charges'].mean():.0f}")
axes[0].axvline(df['monthly_charges'].median(), color='orange', linestyle='--', linewidth=2, label=f"Median: ${df['monthly_charges'].median():.0f}")
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Monthly Charges')
axes[0].legend()

# Right: distribution of tenure
axes[1].hist(df['tenure_months'], bins=20, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Tenure')

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Reading a histogram:")
print("  - X axis: value range")
print("  - Y axis: how many customers fall in each range")
print("  - Tall bar = many customers have that value")
print("  - If mean > median: distribution is right-skewed (outliers pulling mean up)")

In [ ]:
# ── Scatter Plot ─────────────────────────────────────────────────────────────
# Use case: relationship between two numeric features, colored by target

fig, ax = plt.subplots(figsize=(8, 6))

# Plot each class separately for a legend
active  = df[df['churned'] == 0]
churned = df[df['churned'] == 1]

ax.scatter(active['tenure_months'], active['monthly_charges'],
           alpha=0.5, color='steelblue', label='Active', s=30)
ax.scatter(churned['tenure_months'], churned['monthly_charges'],
           alpha=0.7, color='firebrick', label='Churned', s=40, marker='x')

ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.set_title('Tenure vs Monthly Charges (colored by Churn)', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("What do you observe? Are churned customers clustered in a specific region?")

In [ ]:
# ── Bar Chart ────────────────────────────────────────────────────────────────
# Use case: compare a metric across categories

churn_by_contract = df.groupby('contract')['churned'].mean() * 100  # as percentage

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(churn_by_contract.index, churn_by_contract.values,
              color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='black', linewidth=0.5)

# Add value labels on top of bars
for bar, val in zip(bars, churn_by_contract.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Contract Type', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_title('Churn Rate by Contract Type', fontsize=14)
ax.set_ylim(0, max(churn_by_contract.values) * 1.2)

plt.tight_layout()
plt.show()

---

## Section 2: Subplots — Dashboard Layout

Real EDA uses a **grid of charts** to get a complete picture of the data. `plt.subplots(rows, cols)` creates the grid.

In [ ]:
# ── 2x2 EDA Dashboard ────────────────────────────────────────────────────────
# This is the kind of chart grid you'd include in any ML project report.

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Customer Churn EDA Dashboard', fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1 (top-left): distribution of charges by churn status ──
for label, group in df.groupby('churn_label'):
    axes[0, 0].hist(group['monthly_charges'], bins=20, alpha=0.6, label=label, edgecolor='white')
axes[0, 0].set_title('Monthly Charges Distribution by Churn')
axes[0, 0].set_xlabel('Monthly Charges ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# ── Plot 2 (top-right): churn rate by contract type ──
cr = df.groupby('contract')['churned'].mean() * 100
axes[0, 1].bar(cr.index, cr.values, color=sns.color_palette('Set2', len(cr)))
axes[0, 1].set_title('Churn Rate by Contract Type')
axes[0, 1].set_xlabel('Contract')
axes[0, 1].set_ylabel('Churn Rate (%)')
for i, (contract, val) in enumerate(cr.items()):
    axes[0, 1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')

# ── Plot 3 (bottom-left): scatter tenure vs charges ──
scatter = axes[1, 0].scatter(
    df['tenure_months'], df['monthly_charges'],
    c=df['churned'], cmap='RdYlGn_r', alpha=0.5, s=20
)
plt.colorbar(scatter, ax=axes[1, 0], label='Churned')
axes[1, 0].set_title('Tenure vs Charges (color = churn)')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].set_ylabel('Monthly Charges ($)')

# ── Plot 4 (bottom-right): support tickets distribution ──
ticket_means = df.groupby('churned')['support_tickets'].mean()
axes[1, 1].bar(['Active', 'Churned'], ticket_means.values, color=['steelblue', 'firebrick'])
axes[1, 1].set_title('Avg Support Tickets: Active vs Churned')
axes[1, 1].set_ylabel('Avg Tickets')
for i, val in enumerate(ticket_means.values):
    axes[1, 1].text(i, val + 0.05, f'{val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Section 3: seaborn — Statistical Visualization

**seaborn** is built on top of matplotlib but much easier for statistical charts. It understands DataFrames — you just pass column names.

In [ ]:
# ── seaborn Distribution Plots ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# histplot with KDE (combined histogram + smooth density curve)
sns.histplot(data=df, x='monthly_charges', hue='churn_label',
             bins=25, kde=True, alpha=0.5, ax=axes[0])
axes[0].set_title('Monthly Charges by Churn Status (histplot + KDE)')

# kdeplot alone (smooth density, no bars)
sns.kdeplot(data=df, x='tenure_months', hue='churn_label',
            fill=True, alpha=0.4, ax=axes[1])
axes[1].set_title('Tenure Distribution by Churn Status (KDE)')

plt.tight_layout()
plt.show()

print("KDE (Kernel Density Estimate):")
print("  - A smooth curve version of a histogram")
print("  - Shows the 'shape' of a distribution without the bin-size problem")
print("  - The area under the curve always = 1")

In [ ]:
# ── Box Plot ─────────────────────────────────────────────────────────────────
# Shows: median, quartiles (25th and 75th percentile), and outliers

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Monthly charges by contract type
sns.boxplot(data=df, x='contract', y='monthly_charges', hue='churn_label',
            palette='Set2', ax=axes[0])
axes[0].set_title('Monthly Charges by Contract Type and Churn')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Monthly Charges ($)')

# Violin plot (like box plot but shows density shape too)
sns.violinplot(data=df, x='churn_label', y='tenure_months',
               palette='pastel', inner='box', ax=axes[1])
axes[1].set_title('Tenure Distribution by Churn Status (Violin)')
axes[1].set_xlabel('Customer Status')
axes[1].set_ylabel('Tenure (months)')

plt.tight_layout()
plt.show()

print("Box plot anatomy:")
print("  - Center line: median")
print("  - Box edges: 25th and 75th percentile (IQR)")
print("  - Whiskers: 1.5 × IQR from the box edges")
print("  - Dots beyond whiskers: outliers")

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
# Use case: discover which features are correlated with each other AND with the target
# This directly informs feature selection in ML!

numeric_cols = ['tenure_months', 'monthly_charges', 'support_tickets', 'churned']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle (it's symmetric)
sns.heatmap(
    corr_matrix,
    annot=True,      # show values in each cell
    fmt='.2f',       # 2 decimal places
    cmap='RdBu_r',   # red = positive, blue = negative
    vmin=-1, vmax=1, # correlation range
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix — Churn Dataset', fontsize=13)

plt.tight_layout()
plt.show()

print("Reading a correlation matrix:")
print("  +1.0 = perfect positive correlation (as X rises, Y rises)")
print("  -1.0 = perfect negative correlation (as X rises, Y falls)")
print("   0.0 = no linear relationship")
print("\nFocus on the 'churned' row/column — these are your most predictive features!")

In [ ]:
# ── Pair Plot ────────────────────────────────────────────────────────────────
# Use case: all pairwise relationships at once — great for initial EDA
# Warning: slow on large datasets — use on a sample

sample = df.sample(150, random_state=42)  # use a sample for speed

pair_plot = sns.pairplot(
    sample[['tenure_months', 'monthly_charges', 'support_tickets', 'churn_label']],
    hue='churn_label',
    palette={'Active': 'steelblue', 'Churned': 'firebrick'},
    plot_kws={'alpha': 0.5, 's': 25},
    diag_kind='kde'
)
pair_plot.fig.suptitle('Pairplot of Churn Dataset', y=1.02, fontsize=14)
plt.show()

print("Pairplot: each cell shows a scatter plot of two features, colored by class.")
print("Diagonal: the distribution of that feature.")
print("Look for cells where the two classes are clearly separated — these are good features!")

In [ ]:
# ── Categorical Plots ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot: number of customers per contract type, split by churn
sns.countplot(data=df, x='contract', hue='churn_label',
              palette='Set2', ax=axes[0])
axes[0].set_title('Customer Count by Contract and Churn')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Count')
axes[0].legend(title='Status')

# Strip plot: all data points shown (good for small datasets)
sns.stripplot(data=df.sample(100), x='contract', y='monthly_charges',
              hue='churn_label', jitter=True, alpha=0.7,
              palette={'Active': 'steelblue', 'Churned': 'firebrick'}, ax=axes[1])
axes[1].set_title('Charges per Contract Type (Strip Plot)')

plt.tight_layout()
plt.show()

---

## Section 4: Complete EDA Workflow

In [ ]:
# ── Full EDA Report Function ──────────────────────────────────────────────────
# This is the workflow you should follow for EVERY new dataset.

def run_eda(df, target_col, numeric_cols, cat_cols):
    """Run a standard EDA report for a classification dataset."""
    
    print("=" * 60)
    print("EXPLORATORY DATA ANALYSIS REPORT")
    print("=" * 60)
    
    # 1. Shape and dtypes
    print(f"\n📦 Shape: {df.shape}")
    print(f"Missing: {df.isnull().sum().sum()} total")
    
    # 2. Target distribution
    print(f"\n🎯 Target '{target_col}' distribution:")
    vc = df[target_col].value_counts(normalize=True)
    for val, pct in vc.items():
        print(f"   {val}: {pct:.1%}")
    imbalance = vc.max() / vc.min()
    if imbalance > 3:
        print(f"   ⚠️  Class imbalance ratio: {imbalance:.1f}x (may need SMOTE or class_weight)")
    
    # 3. Numeric feature summary
    print(f"\n📊 Numeric features summary:")
    print(df[numeric_cols].describe().round(2))
    
    # 4. Correlation with target
    corrs = df[numeric_cols + [target_col]].corr()[target_col].drop(target_col)
    print(f"\n📈 Correlation with '{target_col}' (|corr| descending):")
    for feat, corr in corrs.abs().sort_values(ascending=False).items():
        direction = "+" if corrs[feat] > 0 else "-"
        bar = "█" * int(abs(corrs[feat]) * 20)
        print(f"   {feat:<22} {direction}{corrs[feat]:.3f}  {bar}")

run_eda(df, 'churned', 
        numeric_cols=['tenure_months', 'monthly_charges', 'support_tickets'],
        cat_cols=['contract'])

---

## Section 5: Saving Figures

In [ ]:
# ── Saving Figures ────────────────────────────────────────────────────────────
# For portfolios, reports, and presentations — always save high-res figures.

os.makedirs('../outputs', exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
churn_by_contract = df.groupby('contract')['churned'].mean() * 100
ax.bar(churn_by_contract.index, churn_by_contract.values,
       color=sns.color_palette('Set2', len(churn_by_contract)))
ax.set_title('Churn Rate by Contract Type', fontsize=14)
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')

# Save options:
fig.savefig('../outputs/churn_by_contract.png', dpi=150, bbox_inches='tight')
fig.savefig('../outputs/churn_by_contract.pdf', bbox_inches='tight')  # vector for papers

plt.show()
print("✅ Figures saved to outputs/")
print("\nsavefig parameters:")
print("  dpi=150   → 150 dots per inch (higher = larger file, better quality)")
print("  bbox_inches='tight' → removes extra whitespace around the figure")

In [ ]:
# ── YOUR TURN: Full EDA ───────────────────────────────────────────────────────
# Use the df dataset to create a 2x2 subplot figure with:
# 1. (top-left)  Histogram of support_tickets (colored by churn_label)
# 2. (top-right) Box plot of monthly_charges by contract type
# 3. (bottom-left) Bar chart: average tenure by contract type  
# 4. (bottom-right) Scatter: support_tickets vs monthly_charges (colored by churned)
# Add titles, axis labels, and a legend to each plot.

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('My EDA Dashboard', fontsize=15, fontweight='bold')

# YOUR CODE HERE

plt.tight_layout()
plt.show()

---

## Chart Type Quick Reference

| Question | Chart Type | Code |
|----------|-----------|------|
| Distribution of one numeric column | Histogram / KDE | `sns.histplot` |
| Distribution split by category | KDE with hue | `sns.kdeplot(hue=...)` |
| Two numeric columns relationship | Scatter plot | `ax.scatter` or `sns.scatterplot` |
| One numeric, one category | Box / Violin | `sns.boxplot`, `sns.violinplot` |
| Count per category | Bar / Count | `ax.bar` or `sns.countplot` |
| All pairwise relationships | Pairplot | `sns.pairplot` |
| Correlations between all features | Heatmap | `sns.heatmap(df.corr())` |
| Metric over time or training | Line plot | `ax.plot` |
| Proportions | Pie chart | `ax.pie` (use rarely) |

---

## Week 4 Reflection Questions

**Answer these (double-click to edit):**

1. What is the difference between `fig` and `ax` in matplotlib?

2. When would you use a histogram vs a box plot?

3. What does a correlation of -0.7 between tenure and churn mean?

4. Why do we look at the correlation heatmap before building an ML model?

5. What does the `hue` parameter do in seaborn?

---

## Week 4 Checklist — Phase 0 Complete!

- [ ] I can create a line plot, scatter plot, bar chart, and histogram with matplotlib
- [ ] I can create a 2x2 grid of subplots
- [ ] I can add titles, axis labels, and legends to every chart
- [ ] I can use `sns.histplot`, `sns.boxplot`, `sns.heatmap`, `sns.pairplot`
- [ ] I understand what a correlation matrix shows
- [ ] I ran every code cell and completed the YOUR TURN dashboard
- [ ] I saved at least one figure to the outputs/ folder
- [ ] I can look at a chart and describe what the data is telling me

---

## 🎉 Phase 0 Complete!

You have completed the foundational ramp:
- **Week 0**: ML concepts, 3 types of ML, 6-step workflow
- **Week 1**: Python basics — variables, lists, dicts, loops, functions, imports
- **Week 2**: pandas — DataFrames, filtering, groupby, cleaning, EDA
- **Week 3**: NumPy — arrays, shapes, broadcasting, dot product
- **Week 4**: Visualization — matplotlib, seaborn, the full EDA workflow

**Next: Phase 1 → `STARTER_Week1_DataExploration.ipynb` → Real ML projects begin.**